In [1]:
!pip install -qU langchain-core langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.3/378.3 kB 26.4 MB/s eta 0:00:00


In [2]:
import os
import re
from typing import Dict, List

from langsmith import traceable
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# -----------------------------
# LangSmith tracing
# -----------------------------
# Keep these if you already entered them earlier
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "AI-Resume-Screening-System"

# -----------------------------
# Step 1: Inputs
# -----------------------------
job_description = """
We are hiring a Data Scientist.

Required Skills:
Python
Machine Learning
Deep Learning
NLP
Statistics
SQL
Data Visualization

Tools:
Pandas
NumPy
Scikit-learn
TensorFlow
Matplotlib
Seaborn

Experience:
Minimum 2+ years experience in data science projects
Experience building ML models
Experience working with real datasets
"""

resume_strong = """
Candidate Name: Rahul Sharma

Skills:
Python, Machine Learning, Deep Learning, NLP, Statistics, SQL, Data Visualization

Tools:
Pandas, NumPy, Scikit-learn, TensorFlow, Matplotlib, Seaborn

Experience:
3 years experience working on ML and NLP projects.
Built multiple predictive models.
Worked on real-world datasets and deployed ML models.
"""

resume_average = """
Candidate Name: Amit Verma

Skills:
Python, Machine Learning, SQL

Tools:
Pandas, NumPy, Matplotlib

Experience:
1 year experience working on academic ML projects.
Basic dataset analysis and visualization experience.
"""

resume_weak = """
Candidate Name: Rohan Singh

Skills:
Excel, PowerPoint, Communication

Tools:
MS Office

Experience:
No experience in data science or machine learning.
"""

# -----------------------------
# Step 2: PromptTemplate (required)
# -----------------------------
extract_prompt = PromptTemplate.from_template("""
You are an AI resume analyzer.

Extract only what is explicitly present in the resume.
Do not assume missing skills.
Return short structured output.

Resume:
{resume}
""")

# -----------------------------
# Step 3: Helper functions
# -----------------------------
CANONICAL_SKILLS = [
    "Python", "Machine Learning", "Deep Learning", "NLP",
    "Statistics", "SQL", "Data Visualization"
]

CANONICAL_TOOLS = [
    "Pandas", "NumPy", "Scikit-learn", "TensorFlow",
    "Matplotlib", "Seaborn", "PyTorch"
]

def normalize_text(text: str) -> str:
    return text.lower().strip()

def extract_years(text: str) -> int:
    match = re.search(r'(\d+)\s*\+?\s*year', text.lower())
    return int(match.group(1)) if match else 0

@traceable(name="skill_extraction")
def extract_resume_info(data: Dict) -> Dict:
    resume = data["resume"]
    formatted_prompt = extract_prompt.invoke({"resume": resume}).to_string()

    found_skills = [s for s in CANONICAL_SKILLS if s.lower() in resume.lower()]
    found_tools = [t for t in CANONICAL_TOOLS if t.lower() in resume.lower()]
    years = extract_years(resume)

    exp_summary = "No relevant experience found."
    exp_match = re.search(r'Experience:(.*)', resume, re.IGNORECASE | re.DOTALL)
    if exp_match:
        exp_summary = exp_match.group(1).strip().replace("\n", " ")

    return {
        "candidate_name": re.search(r'Candidate Name:\s*(.*)', resume).group(1).strip() if re.search(r'Candidate Name:\s*(.*)', resume) else "Unknown",
        "formatted_prompt": formatted_prompt,
        "skills": found_skills,
        "tools": found_tools,
        "years_experience": years,
        "experience_summary": exp_summary,
        "resume": resume,
        "job_description": data["job_description"]
    }

@traceable(name="matching_logic")
def match_resume_to_jd(data: Dict) -> Dict:
    jd = data["job_description"]

    jd_skills = [s for s in CANONICAL_SKILLS if s.lower() in jd.lower()]
    jd_tools = [t for t in CANONICAL_TOOLS if t.lower() in jd.lower()]

    matched_skills = [s for s in data["skills"] if s in jd_skills]
    missing_skills = [s for s in jd_skills if s not in data["skills"]]

    matched_tools = [t for t in data["tools"] if t in jd_tools]
    missing_tools = [t for t in jd_tools if t not in data["tools"]]

    data.update({
        "jd_skills": jd_skills,
        "jd_tools": jd_tools,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "matched_tools": matched_tools,
        "missing_tools": missing_tools,
    })
    return data

@traceable(name="scoring")
def score_candidate(data: Dict) -> Dict:
    skill_score = (len(data["matched_skills"]) / max(len(data["jd_skills"]), 1)) * 60
    tool_score = (len(data["matched_tools"]) / max(len(data["jd_tools"]), 1)) * 20
    exp_score = 20 if data["years_experience"] >= 2 else 10 if data["years_experience"] == 1 else 0

    total_score = round(skill_score + tool_score + exp_score)

    data["fit_score"] = min(total_score, 100)
    return data

@traceable(name="explanation")
def explain_score(data: Dict) -> Dict:
    explanation = f"""
Candidate: {data['candidate_name']}
Fit Score: {data['fit_score']}/100

Matched Skills: {", ".join(data['matched_skills']) if data['matched_skills'] else "None"}
Missing Skills: {", ".join(data['missing_skills']) if data['missing_skills'] else "None"}

Matched Tools: {", ".join(data['matched_tools']) if data['matched_tools'] else "None"}
Missing Tools: {", ".join(data['missing_tools']) if data['missing_tools'] else "None"}

Years of Experience: {data['years_experience']}
Reason:
The score is based on skill match, tool match, and experience match with the job description.
"""
    data["explanation"] = explanation.strip()
    return data

# -----------------------------
# Step 4: LCEL pipeline
# -----------------------------
extract_chain = RunnableLambda(extract_resume_info)
match_chain = RunnableLambda(match_resume_to_jd)
score_chain = RunnableLambda(score_candidate)
explain_chain = RunnableLambda(explain_score)

pipeline = extract_chain | match_chain | score_chain | explain_chain

# -----------------------------
# Step 5: Run 3 candidates (.invoke required)
# -----------------------------
strong_result = pipeline.invoke({
    "resume": resume_strong,
    "job_description": job_description
})

average_result = pipeline.invoke({
    "resume": resume_average,
    "job_description": job_description
})

weak_result = pipeline.invoke({
    "resume": resume_weak,
    "job_description": job_description
})

# -----------------------------
# Step 6: Print outputs
# -----------------------------
print("STRONG CANDIDATE RESULT")
print(strong_result["explanation"])
print("\n" + "="*60 + "\n")

print("AVERAGE CANDIDATE RESULT")
print(average_result["explanation"])
print("\n" + "="*60 + "\n")

print("WEAK CANDIDATE RESULT")
print(weak_result["explanation"])

STRONG CANDIDATE RESULT
Candidate: Rahul Sharma
Fit Score: 100/100

Matched Skills: Python, Machine Learning, Deep Learning, NLP, Statistics, SQL, Data Visualization
Missing Skills: None

Matched Tools: Pandas, NumPy, Scikit-learn, TensorFlow, Matplotlib, Seaborn
Missing Tools: None

Years of Experience: 3
Reason:
The score is based on skill match, tool match, and experience match with the job description.


AVERAGE CANDIDATE RESULT
Candidate: Amit Verma
Fit Score: 46/100

Matched Skills: Python, Machine Learning, SQL
Missing Skills: Deep Learning, NLP, Statistics, Data Visualization

Matched Tools: Pandas, NumPy, Matplotlib
Missing Tools: Scikit-learn, TensorFlow, Seaborn

Years of Experience: 1
Reason:
The score is based on skill match, tool match, and experience match with the job description.


WEAK CANDIDATE RESULT
Candidate: Rohan Singh
Fit Score: 9/100

Matched Skills: Machine Learning
Missing Skills: Python, Deep Learning, NLP, Statistics, SQL, Data Visualization

Matched Tools

/usr/local/lib/python3.12/dist-packages/langsmith/client.py:538: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [3]:
import os
import getpass

os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API Key: ")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "AI-Resume-Screening-System"

Enter LangSmith API Key: ··········


In [4]:
import os

os.environ["LANGSMITH_API_KEY"] = "PASTE_YOUR_KEY_HERE"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "AI-Resume-Screening-System"

In [6]:
strong_result = pipeline.invoke({
    "resume": resume_strong,
    "job_description": job_description
})

In [7]:
import os

os.environ["LANGSMITH_API_KEY"] = "PASTE_YOUR_KEY_HERE"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "AI-Resume-Screening-System"

strong_result = pipeline.invoke({
    "resume": resume_strong,
    "job_description": job_description
})

In [8]:
average_result = pipeline.invoke({
    "resume": resume_average,
    "job_description": job_description
})

weak_result = pipeline.invoke({
    "resume": resume_weak,
    "job_description": job_description
})

## Conclusion

This project implements an AI Resume Screening System using a modular LangChain pipeline.

Pipeline flow:
Resume → Skill Extraction → Matching → Scoring → Explanation

The system evaluates candidates based on:
- skill match
- tool match
- experience match

Fit score is generated between 0–100 along with explanation logic.

## LangChain Concepts Used

- PromptTemplate
- LCEL pipeline
- RunnableLambda
- .invoke() execution method

## LangSmith Tracing

LangSmith tracing was configured using environment variables.
Tracing attempts are visible in the LangSmith dashboard project:
AI-Resume-Screening-System

## Debugging Insight

During tracing with ChatOpenAI model, a RateLimitError (429) occurred due to API quota limitations.
However, the pipeline logic executed correctly using the modular LangChain pipeline structure.

This demonstrates how LangSmith helps monitor and debug pipeline execution.